In [2]:
!pip install -q langchain langchain-community langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 17.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatibl

In [3]:
import sqlite3
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

In [4]:
conn = sqlite3.connect("students.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE students (
    student_id TEXT PRIMARY KEY,
    name TEXT,
    department TEXT,
    python INTEGER,
    database INTEGER,
    ai INTEGER,
    web INTEGER
)
""")

students = [
    ("22CS045", "Dhanushya", "Computer Science", 85, 72, 90, 78),
    ("22CS046", "Rahul", "Computer Science", 65, 70, 68, 72),
    ("22CS047", "Priya", "Information Technology", 92, 88, 95, 90),
    ("22CS048", "Arun", "Information Technology", 55, 60, 58, 62),
    ("22CS049", "Meena", "Computer Science", 78, 85, 80, 88)
]

cursor.executemany("""
INSERT INTO students
VALUES (?, ?, ?, ?, ?, ?, ?)
""", students)

conn.commit()
conn.close()

print("students.db created successfully!")

students.db created successfully!


In [5]:
import sqlite3

conn = sqlite3.connect("students.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM students")

rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

('22CS045', 'Dhanushya', 'Computer Science', 85, 72, 90, 78)
('22CS046', 'Rahul', 'Computer Science', 65, 70, 68, 72)
('22CS047', 'Priya', 'Information Technology', 92, 88, 95, 90)
('22CS048', 'Arun', 'Information Technology', 55, 60, 58, 62)
('22CS049', 'Meena', 'Computer Science', 78, 85, 80, 88)


In [6]:
from langchain_core.tools import tool
import sqlite3

@tool
def get_student_info(student_id: str) -> str:
    """Get the name and department of a student using their student ID."""

    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()

    cursor.execute(
        "SELECT name, department FROM students WHERE student_id = ?",
        (student_id,)
    )

    result = cursor.fetchone()

    conn.close()

    if result:
        name, department = result
        return f"Name: {name}, Department: {department}"

    return "Student not found."

In [7]:
@tool
def get_student_marks(student_id: str) -> str:
    """Get the Python, Database, AI, and Web marks of a student using their student ID."""

    conn = sqlite3.connect("students.db")
    cursor = conn.cursor()

    cursor.execute(
        "SELECT python, database, ai, web FROM students WHERE student_id = ?",
        (student_id,)
    )

    result = cursor.fetchone()

    conn.close()

    if result:
        python, database, ai, web = result
        return f"Python: {python}, Database: {database}, AI: {ai}, Web: {web}"

    return "Student not found."

In [8]:
@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression such as 85+72+90+78 or (85+72+90+78)/4."""

    try:
        result = eval(expression)
        return str(result)
    except:
        return "Invalid mathematical expression."

In [9]:
@tool
def get_passing_rules() -> str:
    """Get the university passing rules for students."""

    return """
    University Passing Rules:
    - Minimum overall average: 40%
    - Minimum mark in each subject: 35%
    """

In [10]:
tools = [
    get_student_info,
    get_student_marks,
    calculator,
    get_passing_rules
]

print("Tools loaded:")
for tool in tools:
    print(tool.name)

Tools loaded:
get_student_info
get_student_marks
calculator
get_passing_rules


In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

print("Gemini model loaded successfully!")

Gemini model loaded successfully!


In [15]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a student information assistant.

    Use the available tools to answer student-related questions.
    Choose the appropriate tools based on the user's question.
    Do not guess student information or marks.
    Use the calculator tool whenever calculations are required.
    Use the passing rules tool when checking whether a student passes.
    """
)

print("Agent created.")

Agent created.


In [16]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the name and department of student 22CS045?"}
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The name of student **22CS045** is **Dhanushya**, and her department is **Computer Science**.', 'extras': {'signature': 'EtoBCtcBAWkUfRPTPgRkKPYNAUcNZJtscKrBYM0XpzvPNA2fHlFHFWOOj572XldGzagD12pBeMzvglGu3xzNMIIHF9XAi89zpfkU0CYM1bP4r894A4BNW5A0LTRs1PsZg6WdfD8hkIGRFmtR/YNFeku26tQQ907AI6d07UzOKJ3qMT36F5lugFnYlnvKkrAeAEYMyvA9cCQIiZMzI/sT13rC1vy3D3vG1hNwG8w8CAJmUuCEf8LFBUGNG3cTxP+lNvvN2c6Rk6SB4JI+b+kJ5cz2Aj6r9aeNpJiez3s='}}]


In [17]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What are the marks of student 22CS047?"}
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The marks for student **22CS047** are as follows:\n\n* **Python:** 92\n* **Database:** 88\n* **AI:** 95\n* **Web:** 90', 'extras': {'signature': 'EtsBCtgBAWkUfROVhV5LNvlaz9gF+DrQOO9kn0APM/Huviy75cBUI6a99WllcgC+Oh3Zz9h2I/uXaNXyNuL439HWio9mi0huhjaeNVNQs8+HX7h7CGD6KAvlyJwpRbbJogRrweGgM2xllDRX/pO/qqG1w6a7IqeAb3ar6OSHq1Bb4kCI8xmE+85mjFT7QvwEIfWuRVAoplsBTHxgZzjIa6pIMZwkpJbT7sj2VpW1zavd62N82ybirygKmW/Opudytjg4Vd4Pv92XT8/GGORlKbQMihn0s6XqvGlwrb/R'}}]


In [18]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the total and average mark of 22CS045?"}
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'For student **22CS045**, the marks obtained are:\n- **Python:** 85\n- **Database:** 72\n- **AI:** 90\n- **Web:** 78\n\n**Calculations:**\n- **Total Mark:** 325\n- **Average Mark:** 81.25', 'extras': {'signature': 'EuABCt0BAWkUfRN/ddpQLtjNK2NVNN28/59s7/zllb3ZlSza/6+zrktfGeYGzIgSFJmigABsGkbsiYwUpdLZ9TZC5WN4wnBAWSItfKcuWHaTVH00MeqHvvmzpxC9aMXmQGP6E+ORjSq2XSq/E019K9AwPTu3NpPfgVmajg2cfIJwnvbCCTh4m0bx7LaeFPa+YGDjNMuUtovwa/QGKuPcS653xEUVjzzPkouI/b33mwBpu5LW5loXLNlB76Vsdb5BOPdPvegoAH8y+0WXFx9aq+Yi6AF0EU/fODMq4WtK0gA+m30='}}]


In [19]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What are the university passing rules?"}
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The university passing rules are as follows:\n\n* **Minimum overall average:** 40%\n* **Minimum mark in each subject:** 35%', 'extras': {'signature': 'Es0BCsoBAWkUfRN3HpzOtQZFCxluxSPMWN5KX4+6H+C6YP8VHnw6MA9a1YCgcuwB18H1Ml570fdwOKBRbQ/xrWhyeqXBS9hvGEOnF8CQ8rf3lQQaPj1s2I+Vd7iDropFRvnAF4rE7ieuCfBp20H+PexUK3JlGnklFz6I4rS8RcYyC3uM8EdQ8oeXCgz2RM5rCSeII40kFbYzO4tyEy106O2aOEHmdDbWVxnN3xYbX+SQQyY5CvEgSnF0UGv1Z5tiaYYXz2opHhE+VWrVEJVdWg=='}}]


In [25]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Is 22CS045 eligible to pass according to the university rules?"}
    ]
})

print(response["messages"][-1].content)

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 1.258035932s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'q

In [20]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "I am 22CS045. Tell me my name, department, total marks, average marks, and whether I satisfy the university passing requirements."
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'Here are your details based on your Student ID (**22CS045**):\n\n* **Name:** Dhanushya\n* **Department:** Computer Science\n* **Total Marks:** 325 / 400\n* **Average Marks:** 81.25%\n\n### Subject Marks:\n* **Python:** 85\n* **Database:** 72\n* **AI:** 90\n* **Web:** 78\n\n---\n\n### Passing Status:\n**Yes, you satisfy the university passing requirements.**\n\n* **Passing Criteria:**\n  1. Minimum mark in each subject: 35% (Your lowest mark is 72)\n  2. Minimum overall average: 40% (Your average is 81.25%)', 'extras': {'signature': 'EvcECvQEAWkUfROuXfML+gDE/S+q04FbrEBh9eMBYjN1zfS7JCvapnOQeCyX0y6zsrBimv4ng2MdufEtkc1fsdY87dYLSNtKdBLsQJWfzUTFV84a46tYmKuep683QnLtblTQ/ANOENOtgTRZOM7KNrGgXLQOPLvIohm8HfPVVtDqQoxmVo/HEUcFLzfyrBTNb4dGUpAXPuQW2LdGBptBVBfc0UcLEbqSEaCG9q1KUZOtK6g2W1GSLTY1AxrskG0RSrPkRs3F0CEzneC8501TEKiKEdHlH86/76lJtkVYjGD5YByHn+s8jiY2dlczuXZRi4t76zeDtt8dfPBjhlvDNnQgGQ2QmKwhnI/ien4fSMIk1/H7px/nImWrgsqIXpOeQNXzwKw6bcZq+YcScmH4Gwlon+5Ik+dHYCtx7PJ7BYxj6SA/bA

In [22]:
for message in response["messages"]:
    print(type(message).__name__)
    print(message.content)
    print("-" * 50)

HumanMessage
I am 22CS045. Tell me my name, department, total marks, average marks, and whether I satisfy the university passing requirements.
--------------------------------------------------
AIMessage
[]
--------------------------------------------------
ToolMessage
Name: Dhanushya, Department: Computer Science
--------------------------------------------------
ToolMessage
Python: 85, Database: 72, AI: 90, Web: 78
--------------------------------------------------
ToolMessage

    University Passing Rules:
    - Minimum overall average: 40%
    - Minimum mark in each subject: 35%
    
--------------------------------------------------
AIMessage
[]
--------------------------------------------------
ToolMessage
325
--------------------------------------------------
ToolMessage
81.25
--------------------------------------------------
AIMessage
[{'type': 'text', 'text': 'Here are your details based on your Student ID (**22CS045**):\n\n* **Name:** Dhanushya\n* **Department:** Computer Sc

In [23]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the total mark of 22CS047?"
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The'}, {'type': 'text', 'text': ' marks for **Priya** (ID: **22CS047**), Department of Information Technology, are as follows:\n\n- **Python:** 92\n- **Database:** 88\n- **AI:** 95\n- **Web:** 90\n\n**Total Mark:** **365**', 'extras': {'signature': 'EroCCrcCAWkUfRND91Z3UDQw0PqRlzcmGD/pVAOr5k7yNOfMnRQVD9dU+b+X0kJ2UNDznc2YbMNhVRMK7iK0AN3wkIRsp51vMGfpRTO5QWVxfV8zRv90644xPdftjjkxp5wjOUTJW1BKtT54V1bT4JjW1eFY5NVhOlIr8ziEQSbefsQC8P7K6+n6reiE0oTy1YRpNZXxoyuKx8USOaadTz5mX0MZAZdygcF/a2xFNRJEcpuZIE2buZQHKSwH7rLE0ToGBtalhQGVwc1tqnU0vQ2+ej/zRDnModP3BwRNA2N72voo+hy+pYNP2sJp6VByWXKXjmFJykhH0V57v7agR/X33Kj4q9QC4OlFq30eEkIdf1s3c1gTH0NQKni8Ilfp5Im2pZAL/HWhzsGhTHkfxaFylf3EosTe+LtzuCk='}}]


In [24]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the name and department of student 22CS999?"
        }
    ]
})

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'No student was found with the ID **22CS999**. Please check the student ID and try again.', 'extras': {'signature': 'EtcBCtQBAWkUfRNxRVqYWZ2QxyAT8hRMyH1r8SfDg0RovjLN5yNPB7UhHWnhrsTs7eV8J/reYyaKFUFcbAdxXCGytIijzXUwl7MPTDp7TvHNUhSxfiepRcQv8zYuHzdDzbrwI0ubSLgrr1G0BMyrWen32UxnaH/K+aKnPibzxICNNt35Qk8lIsykgyflyw4KV1Tltdn9KFacITOJUkLeU0iHqz9Lc0pAeF8eifYTiHfepXbipGKYt1LN1l3z0CSFyU3fyclMr48iuIPpn5kq1p+po7tEUldZnKM='}}]
